## Caching

Caching is a technique used to temporarily store a copies if data in a high-speed storage layers(RAM) toreduce the time taken to access the data.
The primary goal of caching is to improve system performance by reducing latency, offloading the main data store, and providing faster data retrieval.

Caching is essential for the following reasons:

1. Improved Performance: By storing frequently accessed data in a cache, the time required to retrieve that data is significantly reduced.
2. Reduced Load on Backend Systems: Caching reduces the number of requests that need to be processed by the backend, freeing up resources for other operations.
3. Increased Scalability: Caches help in handling a large number of read requests, making the system more scalable.
4. Cost Efficiency: By reducing the load on backend systems, caching can help lower infrastructure costs.
5. Enhanced User Experience: Faster response times lead to a better user experience, particularly for web and mobile applications.

### Types of Caching

1. In-memory Cache: In-memory caches store data in the main memory (RAM) for extremely fast access.
These caches are typically used for session management, storing frequently accessed objects, and as a front for databases.
Examples: Redis and Memcached.

2. Distributed Cache: A distributed cache spans multiple servers and is designed to handle large-scale systems. It ensures that cached data is available across different nodes in a distributed system.
Ex: Redis Cluster and Amazon ElastiCache

3. Client-side Cache: Client-side caching involves storing data on the client device, typically in the form of cookies, local storage, or application-specific caches. This is commonly used in web browsers to cache static assets like images, scripts, and stylesheets.

4. Database Cache: Database caching involves storing frequently queried database results in a cache.
This reduces the number of queries made to the database, improving performance and scalability.

5. Content Delivery Network(CDN): CDN is used to store copies of content on servers distributed across different geographical locations.This reduces the number of queries made to the database, improving performance and scalability.

### Caching Strategies
1. Cache-Aside (Lazy Loading): The application is responsible for reading and writing from both the cache and the database. Most commonly used. Highly recommand for interviews.

2. Read-Through Cache: The application first checks the cache for data. If it's not there (a cache miss), it retrieves the data from the database and updates the cache. Good for read heavy applications.
It is same as cache aside only difference is cache acts as proxy.

3. Write-Through Cache: Data is written to both the cache and the database simultaneously, ensuring consistency but potentially impacting write performance.

4. Write-Back/write-behind Cache: Data is written to the cache first and later synchronized with the database, improving write performance but risking data loss. We can mitigate it by using persistent caching solution like Redis with AOF(Append only files). This is ideal for write heavy scenarioes where write operations need to be fast and frequent, but immediate consistency with the database is not critical.

6. Write-Around: The data is written directly to the database, bypassing the cache. The cache is only updated when data is requested later during read operation, at which point the cache aside strategy is used to load the data into the cache.

### Cache Eviction Policies
To manage the limited size of a cache, eviction policies are used to determine which data should be removed when the cache is full. Only remember first 4 policies.

1. Least Recently Used (LRU): LRU evicts the least recently accessed data when the cache is full. It assumes that recently used data will likely be used again soon.

2. Least Frequently Used (LFU): LFU evicts data that has been accessed the least number of times, under the assumption that rarely accessed data is less likely to be needed.

3. First In, First Out (FIFO): FIFO evicts the oldest data in the cache first, regardless of how often or recently it has been accessed.

4. Time-to-Live (TTL): TTL is a time-based eviction policy where data is removed from the cache after a specified duration, regardless of usage.

5. Most Recently Used (MRU): MRU is the opposite of Least Recently Used (LRU). In MRU, the item that was accessed most recently is the first to be evicted when the cache is full.
The idea behind MRU is that the most recently accessed item is likely to be a temporary need and won’t be accessed again soon, so evicting it frees up space for potentially more valuable data.

6. Random Replacement (RR): RR cache eviction strategy is the simplest of all: when the cache is full, it evicts a random item to make space for a new one. It doesn't track recency, frequency, or insertion order, making it a lightweight approach with minimal computational overhead.

7. Two-Tiered Caching: Two-Tiered Caching combines two layers of cache—usually a local cache (in-memory) and a remote cache (distributed or shared). The local cache serves as the first layer (hot cache), providing ultra-fast access to frequently used data, while the remote cache acts as the second layer (cold cache) for items not found in the local cache but still needed relatively quickly.

Ex: Local cache resides in HashMap or LRUCache in the application where as remote cache stays in Redis, Memcached.

### Challenges and Considerations
1. Cache Coherence: Ensuring that data in the cache remains consistent with the source of truth (e.g., the database).

2. Cache Invalidation: Determining when and how to update or remove stale data from the cache.

3. Cold Start: Handling scenarios when the cache is empty, such as after a system restart.

4. Cache Eviction Policies: Deciding which items to remove when the cache reaches capacity (e.g., Least Recently Used, Least Frequently Used).

5. Cache Penetration: Preventing malicious attempts to repeatedly query for non-existent data, potentially overwhelming the backend.

6. Cache Stampede: Managing situations where many concurrent requests attempt to rebuild the cache simultaneously.

### Best Practices for Implementing Caching
1. Cache the Right Data: Focus on caching data that is expensive to compute or retrieve and that is frequently accessed.

2. Set Appropriate TTLs: Use TTLs to automatically invalidate cache entries and prevent stale data.

3. Consider Cache Warming: Preload essential data into the cache to avoid cold starts.

4. Monitor Cache Performance: Regularly monitor cache hit/miss ratios and adjust caching strategies based on usage patterns.

5. Use Layered Caching: Implement caching at multiple layers (e.g., client-side, server-side, CDN) to maximize performance benefits.

6. Handle Cache Misses Gracefully: Ensure that the system can handle cache misses efficiently without significant performance degradation.

### Common Issues in Cache

#### Cache Stampede (Thundering Herd)

A cache stampede happens when a popular cache entry expires and many concurrent requests
all get a cache miss at the same moment — each one then hits the database to rebuild the
same entry simultaneously.

```
t=0   key "product:123" is cached
t=60s TTL expires
t=60s 500 requests arrive simultaneously
        → all 500 get cache miss
        → all 500 query the database
        → all 500 compute the same result
        → all 500 write to cache (499 writes are wasted)
        → database is briefly overwhelmed
```

The more popular the key, the worse the stampede.

---

#### Three solutions

**1. Locking (Mutex)**
Only one request rebuilds the cache. All others wait for it to finish, then read the freshly populated value.

```
Request 1  →  cache miss  →  acquires lock  →  queries DB  →  writes cache  →  releases lock
Request 2  →  cache miss  →  waits for lock  →  reads cache (populated by Request 1)
Request 3  →  cache miss  →  waits for lock  →  reads cache
...
Request 500  →  same
```
Downside: all waiting requests are blocked — if the DB query is slow, latency spikes.

---

**2. Probabilistic Early Expiry (XFetch)**
Before the key actually expires, a small random fraction of requests treat it as already
expired and proactively refresh it in the background. The probability of triggering a
refresh increases as the TTL approaches zero.

```python
# Refresh if: -beta * log(random()) * compute_time > time_remaining
# beta=1 is standard; higher beta = more aggressive early refresh
```
No lock needed. The cache is refreshed once (or a few times) before expiry, so no stampede ever starts.

---

**3. Stale-While-Revalidate**
Serve the stale (expired) value immediately to all callers, while exactly one background
worker silently refreshes the cache. Zero latency spike, no waiting.

```
key expires
Request 1  →  serves stale value  →  triggers background refresh
Request 2  →  serves stale value  (refresh still in progress)
Request 3  →  serves stale value  (refresh still in progress)
background →  DB query completes  →  writes fresh value to cache
Request 4  →  serves fresh value
```
Downside: callers briefly see stale data — only acceptable when slight staleness is tolerable.

---

#### Comparison

| Solution | Blocks callers? | Stale data served? | Complexity |
|---|---|---|---|
| Locking (Mutex) | Yes — waiters block | No | Medium |
| Probabilistic Early Expiry | No | No | Low |
| Stale-While-Revalidate | No | Yes (briefly) | Low |


In [ ]:
import time
import math
import random
import threading
from dataclasses import dataclass, field


def slow_db_fetch(key: str) -> str:
    """Simulates an expensive database query."""
    time.sleep(0.1)
    return f"db_value_for_{key}"


# ---------------------------------------------------------------------------
# 1. Locking (Mutex)
# ---------------------------------------------------------------------------
class MutexCache:
    """
    Cache miss → one thread rebuilds, all others wait.
    Lock is per-key so unrelated keys don't block each other.
    """

    def __init__(self, ttl: float):
        self.ttl = ttl
        self._store: dict[str, tuple[str, float]] = {}   # key → (value, expiry)
        self._key_locks: dict[str, threading.Lock] = {}
        self._meta_lock = threading.Lock()

    def _get_key_lock(self, key: str) -> threading.Lock:
        with self._meta_lock:
            if key not in self._key_locks:
                self._key_locks[key] = threading.Lock()
            return self._key_locks[key]

    def get(self, key: str) -> str:
        # Fast path: cache hit (no lock needed for read)
        if key in self._store:
            value, expiry = self._store[key]
            if time.monotonic() < expiry:
                return value

        # Slow path: acquire per-key lock, recheck, then rebuild
        lock = self._get_key_lock(key)
        with lock:
            # Recheck after acquiring lock — another thread may have rebuilt already
            if key in self._store:
                value, expiry = self._store[key]
                if time.monotonic() < expiry:
                    return value
            value = slow_db_fetch(key)
            self._store[key] = (value, time.monotonic() + self.ttl)
            return value


# ---------------------------------------------------------------------------
# 2. Probabilistic Early Expiry (XFetch)
# ---------------------------------------------------------------------------
@dataclass
class XFetchEntry:
    value: str
    expiry: float
    compute_time: float   # how long the last DB fetch took (in seconds)


class XFetchCache:
    """
    Proactively refreshes the cache before TTL expires using:
        refresh if: -beta * log(random()) * compute_time > time_remaining

    As time_remaining → 0, the right-hand side shrinks and the probability
    of triggering a refresh increases. No lock, no stale data.
    """

    def __init__(self, ttl: float, beta: float = 1.0):
        self.ttl = ttl
        self.beta = beta
        self._store: dict[str, XFetchEntry] = {}

    def _should_refresh(self, entry: XFetchEntry) -> bool:
        time_remaining = entry.expiry - time.monotonic()
        # XFetch formula — probability rises as expiry approaches
        return -self.beta * math.log(random.random()) * entry.compute_time > time_remaining

    def get(self, key: str) -> str:
        if key in self._store and not self._should_refresh(self._store[key]):
            return self._store[key].value

        # Refresh
        t0 = time.monotonic()
        value = slow_db_fetch(key)
        compute_time = time.monotonic() - t0
        self._store[key] = XFetchEntry(
            value=value,
            expiry=time.monotonic() + self.ttl,
            compute_time=compute_time,
        )
        return value


# ---------------------------------------------------------------------------
# 3. Stale-While-Revalidate
# ---------------------------------------------------------------------------
@dataclass
class StaleEntry:
    value: str
    expiry: float
    refreshing: bool = False


class StaleWhileRevalidateCache:
    """
    Expired key → serve stale value immediately, refresh in background.
    Only one background refresh fires per key (guarded by `refreshing` flag).
    """

    def __init__(self, ttl: float):
        self.ttl = ttl
        self._store: dict[str, StaleEntry] = {}
        self._lock = threading.Lock()

    def _refresh(self, key: str) -> None:
        value = slow_db_fetch(key)
        with self._lock:
            self._store[key] = StaleEntry(value=value, expiry=time.monotonic() + self.ttl)

    def get(self, key: str) -> tuple[str, bool]:
        """Returns (value, is_stale)."""
        with self._lock:
            entry = self._store.get(key)
            now = time.monotonic()

            if entry is None:
                # Cold miss — block once to populate (no stale value to serve)
                entry = StaleEntry(value="", expiry=0, refreshing=True)
                self._store[key] = entry

            if now >= entry.expiry and not entry.refreshing:
                # Stale — serve old value, kick off background refresh
                entry.refreshing = True
                threading.Thread(target=self._refresh, args=(key,), daemon=True).start()
                return entry.value, True

        if entry.value == "":
            # First ever request — must wait for DB
            self._refresh(key)
            return self._store[key].value, False

        return entry.value, now >= entry.expiry


# ---------------------------------------------------------------------------
# Demo — simulate 10 concurrent requests on an expired key
# ---------------------------------------------------------------------------
def simulate(cache, label: str, key: str = "product:123"):
    db_call_count = [0]
    original_fetch = slow_db_fetch

    results = []
    threads = []

    def request(req_id: int):
        t0 = time.monotonic()
        if isinstance(cache, StaleWhileRevalidateCache):
            value, stale = cache.get(key)
            results.append((req_id, value, round((time.monotonic() - t0) * 1000), stale))
        else:
            value = cache.get(key)
            results.append((req_id, value, round((time.monotonic() - t0) * 1000)))

    for i in range(1, 11):
        t = threading.Thread(target=request, args=(i,))
        threads.append(t)

    print(f"\n=== {label} ===")
    print(f"  {'Req':<5} {'Value':<25} {'Latency (ms)'}")
    print(f"  {'-' * 50}")

    for t in threads:
        t.start()
    for t in threads:
        t.join()

    results.sort(key=lambda r: r[0])
    for r in results:
        if len(r) == 4:
            req_id, value, latency, stale = r
            note = " (stale)" if stale else ""
            print(f"  {req_id:<5} {value:<25} {latency}ms{note}")
        else:
            req_id, value, latency = r
            print(f"  {req_id:<5} {value:<25} {latency}ms")


# Short TTL so it expires immediately for demo purposes
simulate(MutexCache(ttl=0.001),               "Locking (Mutex)")
simulate(XFetchCache(ttl=10.0, beta=100.0),   "Probabilistic Early Expiry (XFetch) — beta=100 forces refresh")
simulate(StaleWhileRevalidateCache(ttl=0.001),"Stale-While-Revalidate")


#### Cache Consistency

Cache consistency is the problem of keeping the cache and the database in sync when data is written.
A read always goes to the cache first — the challenge is what happens when a **write** occurs.

There are four write strategies, each making a different trade-off between consistency, latency, and durability.

---

##### 1. Write-Through

Write to **cache and database simultaneously** in the same operation.

```
Client  →  write(key, value)
              ↓
         Cache ← value      (updated)
         DB    ← value      (updated)
              ↓
         ack to client
```

- **Consistency**: strong — cache and DB are always in sync after every write.
- **Write latency**: higher — must wait for both cache and DB to confirm.
- **Read latency**: fast — cache always has the latest value.
- **Best for**: read-heavy workloads where stale reads are unacceptable (e.g. user profiles, account balances).

---

##### 2. Write-Back (Write-Behind)

Write to the **cache only**. DB is updated asynchronously in the background.

```
Client  →  write(key, value)
              ↓
         Cache ← value      (updated immediately)
         ack to client      (fast)
              ↓  (background, after delay)
         DB    ← value      (updated later)
```

- **Consistency**: eventual — DB lags behind cache until the background flush runs.
- **Write latency**: very low — client only waits for the cache write.
- **Risk**: if the cache crashes before flushing, writes are lost. Mitigated by Redis AOF.
- **Best for**: write-heavy workloads where speed matters more than strict durability (e.g. analytics counters, real-time metrics).

---

##### 3. Write-Around

Write **directly to the database**, bypassing the cache entirely.
Cache is only populated on the next read (cache-aside pattern).

```
Client  →  write(key, value)
              ↓
         DB    ← value      (updated)
         Cache  (untouched — still has old value or nothing)
              ↓  (next read)
         cache miss → DB read → cache populated
```

- **Consistency**: eventual — cache serves stale data until the TTL expires or entry is explicitly invalidated.
- **Write latency**: low — only one write to DB.
- **Best for**: data that is written once and read rarely (e.g. log entries, historical records, large media metadata).

---

##### 4. Cache Invalidation

Instead of updating the cache on write, **delete the cache entry**.
The next read will repopulate it from the DB.

```
Client  →  write(key, value)
              ↓
         DB    ← value      (updated)
         Cache.delete(key)  (invalidated)
              ↓  (next read)
         cache miss → DB read → cache repopulated with fresh value
```

- **Consistency**: strong for reads after the first miss — no stale value is ever served.
- **Write latency**: low — delete is cheap.
- **Risk**: thundering herd on popular keys (see Cache Stampede above).
- **Best for**: general-purpose — the default choice when in doubt.

---

##### The "Update or Invalidate?" decision

When a write happens, you have two cache options: **update the cached value** or **delete it**.

| | Update cache | Delete (invalidate) cache |
|---|---|---|
| Next read | Cache hit (no DB round-trip) | Cache miss (one DB round-trip) |
| Stale data risk | Yes — if update and DB write race | No — miss forces a fresh DB read |
| Complexity | Higher — must keep cache value correct | Lower — delete is always safe |

**Invalidation is almost always safer.** Updating the cache requires the write path to compute the exact new cached representation, which can be complex (e.g. denormalised aggregates). Deletion is always correct — the worst case is one extra DB read.

---

##### Race condition: Write then Invalidate vs Invalidate then Write

Order of operations matters. Consider two concurrent operations:

```
Wrong order (Invalidate → Write):
  Thread A: invalidates cache
  Thread B: reads cache miss → fetches old value from DB → repopulates cache
  Thread A: writes new value to DB
  Result: cache now holds the OLD value  ← stale

Correct order (Write → Invalidate):
  Thread A: writes new value to DB
  Thread A: invalidates cache
  Thread B: reads cache miss → fetches NEW value from DB → repopulates cache
  Result: cache holds the NEW value  ← consistent
```

**Always write to DB first, then invalidate the cache.**


In [ ]:
import time
import threading
from collections import defaultdict


def db_read(key: str) -> str:
    time.sleep(0.05)   # simulate DB latency
    return f"db_value:{key}"

def db_write(key: str, value: str) -> None:
    time.sleep(0.05)   # simulate DB latency


# ---------------------------------------------------------------------------
# 1. Write-Through
# ---------------------------------------------------------------------------
class WriteThroughCache:
    def __init__(self):
        self._cache: dict[str, str] = {}
        self._db: dict[str, str] = {}

    def write(self, key: str, value: str) -> None:
        db_write(key, value)          # write DB first
        self._cache[key] = value      # then update cache
        self._db[key] = value

    def read(self, key: str) -> str:
        if key in self._cache:
            return self._cache[key]   # cache hit
        value = db_read(key)          # cache miss → DB
        self._cache[key] = value
        return value

    def state(self) -> dict:
        return {"cache": dict(self._cache), "db": dict(self._db)}


# ---------------------------------------------------------------------------
# 2. Write-Back (Write-Behind)
# ---------------------------------------------------------------------------
class WriteBackCache:
    def __init__(self, flush_interval: float = 0.2):
        self._cache: dict[str, str] = {}
        self._db: dict[str, str] = {}
        self._dirty: set[str] = set()   # keys not yet flushed to DB
        self._lock = threading.Lock()
        # Background flush thread
        self._timer = threading.Timer(flush_interval, self._flush)
        self._timer.daemon = True
        self._timer.start()

    def _flush(self) -> None:
        with self._lock:
            for key in list(self._dirty):
                self._db[key] = self._cache[key]
                db_write(key, self._cache[key])
            self._dirty.clear()

    def write(self, key: str, value: str) -> None:
        with self._lock:
            self._cache[key] = value    # only write to cache
            self._dirty.add(key)        # mark for background flush

    def read(self, key: str) -> str:
        with self._lock:
            if key in self._cache:
                return self._cache[key]
        value = db_read(key)
        with self._lock:
            self._cache[key] = value
        return value

    def state(self) -> dict:
        with self._lock:
            return {
                "cache": dict(self._cache),
                "db": dict(self._db),
                "dirty_keys": list(self._dirty),
            }


# ---------------------------------------------------------------------------
# 3. Write-Around
# ---------------------------------------------------------------------------
class WriteAroundCache:
    def __init__(self):
        self._cache: dict[str, str] = {}
        self._db: dict[str, str] = {}

    def write(self, key: str, value: str) -> None:
        db_write(key, value)            # write DB only, skip cache
        self._db[key] = value
        self._cache.pop(key, None)      # invalidate stale cache entry if present

    def read(self, key: str) -> str:
        if key in self._cache:
            return self._cache[key]     # cache hit
        value = self._db.get(key) or db_read(key)
        self._cache[key] = value        # populate on read
        return value

    def state(self) -> dict:
        return {"cache": dict(self._cache), "db": dict(self._db)}


# ---------------------------------------------------------------------------
# 4. Cache Invalidation
# ---------------------------------------------------------------------------
class CacheInvalidation:
    def __init__(self):
        self._cache: dict[str, str] = {}
        self._db: dict[str, str] = {}

    def write(self, key: str, value: str) -> None:
        db_write(key, value)            # 1. write DB first
        self._db[key] = value
        del self._cache[key] if key in self._cache else None   # 2. then invalidate

    def read(self, key: str) -> str:
        if key in self._cache:
            return self._cache[key]     # cache hit
        value = self._db.get(key) or db_read(key)
        self._cache[key] = value        # repopulate on miss
        return value

    def state(self) -> dict:
        return {"cache": dict(self._cache), "db": dict(self._db)}


# ---------------------------------------------------------------------------
# Demo
# ---------------------------------------------------------------------------
def run_demo(cache, label: str):
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")

    t0 = time.monotonic()
    cache.write("user:1", "alice")
    write_ms = round((time.monotonic() - t0) * 1000)
    print(f"  write('user:1', 'alice')   → {write_ms}ms")
    print(f"  state after write  : {cache.state()}")

    t0 = time.monotonic()
    val = cache.read("user:1")
    read_ms = round((time.monotonic() - t0) * 1000)
    print(f"  read('user:1')             → '{val}'  ({read_ms}ms)")

    # Update the value
    cache.write("user:1", "alice_updated")
    print(f"  write('user:1', 'alice_updated')")
    print(f"  state after update : {cache.state()}")

    val = cache.read("user:1")
    print(f"  read('user:1')             → '{val}'")


run_demo(WriteThroughCache(),  "1. Write-Through")
run_demo(WriteAroundCache(),   "3. Write-Around")
run_demo(CacheInvalidation(),  "4. Cache Invalidation")

# Write-Back needs a moment to flush
wb = WriteBackCache(flush_interval=0.3)
print(f"\n{'='*55}")
print(f"  2. Write-Back (Write-Behind)")
print(f"{'='*55}")
wb.write("user:1", "alice")
print(f"  write('user:1', 'alice')   → immediate (async)")
print(f"  state right after  : {wb.state()}  ← DB not yet updated")
time.sleep(0.4)
print(f"  state after flush  : {wb.state()}  ← background flush completed")

# ---------------------------------------------------------------------------
# Race condition demo — wrong order vs correct order
# ---------------------------------------------------------------------------
print(f"\n{'='*55}")
print("  Race condition: Invalidate→Write (WRONG) vs Write→Invalidate (RIGHT)")
print(f"{'='*55}")

cache = {"user:1": "old_alice"}
db    = {"user:1": "old_alice"}

# Wrong order
cache["user:1_wrong"] = "old_alice"
del cache["user:1_wrong"]            # step 1: invalidate
# concurrent read repopulates from DB before DB is updated
stale_value = db.get("user:1")       # still old value
cache["user:1_wrong"] = stale_value  # repopulated with stale
db["user:1"] = "new_alice"           # step 2: write DB (too late)
print(f"  Wrong order → cache='{cache.get('user:1_wrong')}'  db='{db['user:1']}'  ← mismatch")

# Correct order
db["user:1"] = "new_alice"           # step 1: write DB first
del cache["user:1"]                  # step 2: invalidate
# next read fetches fresh value
cache["user:1"] = db["user:1"]
print(f"  Right order → cache='{cache['user:1']}'  db='{db['user:1']}'  ← consistent")


#### Hot Key Issue

A hot key is a single cache key that receives a disproportionately large share of traffic.
In a distributed cache (e.g. Redis Cluster), each key lives on exactly one shard.
If that key is extremely popular, all reads funnel to one shard — saturating its CPU,
memory bandwidth, or network — while all other shards sit idle.

```
                    ┌─────────────┐
                    │   Shard A   │  ← "product:iphone15" lives here
100,000 req/sec ───→│  HOT KEY    │  ← CPU 100%, network saturated
                    └─────────────┘

                    ┌─────────────┐
         ~10 req/s  │   Shard B   │  ← idle
                    └─────────────┘

                    ┌─────────────┐
         ~10 req/s  │   Shard C   │  ← idle
                    └─────────────┘
```

---

#### Why it happens

Hot keys naturally occur around:
- **Viral content** — a trending post, a product launch, a breaking news article
- **Global config** — feature flags, rate limit rules read on every request
- **Celebrity data** — a famous user's profile or follower count
- **Flash sales** — a single product SKU during a sale event

---

#### Four solutions

**1. Key Replication (Read Replicas)**

Store the same value under multiple keys, spread across different shards.
The client picks one at random on each read — load is distributed across N shards.

```
"product:iphone15"          → Shard A
"product:iphone15:replica:1" → Shard B
"product:iphone15:replica:2" → Shard C

read → pick random replica key → each shard handles ~1/N of the traffic
```

Write must update all replicas. Acceptable for read-heavy, rarely-written hot keys.

---

**2. Local In-Process Cache (L1 Cache)**

Each application server maintains a tiny in-memory dict (or LRU cache) for the
hottest keys. Requests never leave the process for those keys — Redis is not touched.

```
Request arrives at App Server 1
    ↓
Local cache hit?  YES → return immediately  (no Redis, no network)
                  NO  → fetch from Redis → populate local cache → return
```

Consistency trade-off: local cache has its own TTL, so different servers may briefly
serve slightly different values. Acceptable for data that changes infrequently.

---

**3. Key Hashing with Virtual Shards**

Add a random suffix to the key when writing, spreading it across multiple real slots.
Reads must query all suffixed variants and merge results (or just pick one randomly).

```
write "product:iphone15" → write to "product:iphone15#0", "#1", "#2", "#3"

read  → pick random suffix → "product:iphone15#2" → different shard each time
```

Works well for simple scalar values (counters, strings). Harder for complex objects
that must stay consistent across all shards.

---

**4. Request Coalescing**

Multiple concurrent requests for the same key are collapsed into one upstream fetch.
All waiting requests share the single result when it returns.

```
Requests 1–100 all miss "product:iphone15"
    ↓
Request 1 acquires a per-key in-flight lock and fetches from Redis/DB
Requests 2–100 wait on the same future/event
    ↓
Result arrives → all 100 requests resolved at once
    ↓
Only 1 upstream call was made instead of 100
```

This is different from Mutex Cache (which still hits Redis once per caller).
Coalescing collapses N callers into 1 upstream call.

---

#### Comparison

| Solution | Reduces Redis load? | Consistency | Best for |
|---|---|---|---|
| Key Replication | Yes — spread across shards | Eventual (replica lag) | Read-heavy, rarely written |
| Local L1 Cache | Yes — skips Redis entirely | Eventual (local TTL) | Global config, feature flags |
| Virtual Shards | Yes — spread across slots | Depends on merge strategy | Counters, scalar values |
| Request Coalescing | Yes — N callers → 1 fetch | Strong | Burst reads on the same key |


In [ ]:
import time
import random
import threading
from collections import OrderedDict


def fetch_from_redis(key: str) -> str:
    """Simulates a Redis round-trip."""
    time.sleep(0.05)
    return f"value:{key}"


# ---------------------------------------------------------------------------
# 1. Key Replication
# ---------------------------------------------------------------------------
class KeyReplicationCache:
    """
    Writes to N replica keys spread across virtual shards.
    Reads pick a random replica — each shard handles ~1/N of traffic.
    """

    def __init__(self, replicas: int = 3):
        self.replicas = replicas
        self._store: dict[str, str] = {}

    def _replica_keys(self, key: str) -> list[str]:
        return [f"{key}:replica:{i}" for i in range(self.replicas)]

    def write(self, key: str, value: str) -> None:
        for rk in self._replica_keys(key):
            self._store[rk] = value

    def read(self, key: str) -> str:
        rk = random.choice(self._replica_keys(key))
        return self._store.get(rk) or fetch_from_redis(rk)

    def shard_distribution(self, key: str, reads: int = 100) -> dict[str, int]:
        counts: dict[str, int] = {rk: 0 for rk in self._replica_keys(key)}
        for _ in range(reads):
            counts[random.choice(self._replica_keys(key))] += 1
        return counts


# ---------------------------------------------------------------------------
# 2. Local In-Process (L1) Cache
# ---------------------------------------------------------------------------
class LRUCache:
    """Minimal LRU cache using OrderedDict."""

    def __init__(self, capacity: int):
        self.capacity = capacity
        self._cache: OrderedDict[str, tuple[str, float]] = OrderedDict()

    def get(self, key: str, ttl: float) -> str | None:
        if key not in self._cache:
            return None
        value, expiry = self._cache[key]
        if time.monotonic() > expiry:
            del self._cache[key]
            return None
        self._cache.move_to_end(key)
        return value

    def set(self, key: str, value: str, ttl: float) -> None:
        self._cache[key] = (value, time.monotonic() + ttl)
        self._cache.move_to_end(key)
        if len(self._cache) > self.capacity:
            self._cache.popitem(last=False)


class LocalL1Cache:
    """
    Two-layer cache: local in-process LRU (L1) → Redis (L2).
    Hot keys are served from L1 — Redis is never touched for those reads.
    """

    def __init__(self, l1_capacity: int = 100, l1_ttl: float = 5.0):
        self.l1 = LRUCache(l1_capacity)
        self.l1_ttl = l1_ttl
        self._redis: dict[str, str] = {}   # simulated Redis

    def write(self, key: str, value: str) -> None:
        self._redis[key] = value
        # Invalidate L1 on write to avoid serving stale data
        self.l1._cache.pop(key, None)

    def read(self, key: str) -> tuple[str, str]:
        # L1 hit
        val = self.l1.get(key, self.l1_ttl)
        if val is not None:
            return val, "L1"
        # L2 (Redis) hit
        val = self._redis.get(key) or fetch_from_redis(key)
        self.l1.set(key, val, self.l1_ttl)
        return val, "L2(Redis)"


# ---------------------------------------------------------------------------
# 3. Virtual Shards (key suffix scatter)
# ---------------------------------------------------------------------------
class VirtualShardCache:
    """
    Scatters one logical key across N virtual shards via numeric suffix.
    Read picks a random shard. Write updates all shards.
    """

    def __init__(self, shards: int = 4):
        self.shards = shards
        self._store: dict[str, str] = {}

    def _shard_key(self, key: str, shard: int) -> str:
        return f"{key}#{shard}"

    def write(self, key: str, value: str) -> None:
        for i in range(self.shards):
            self._store[self._shard_key(key, i)] = value

    def read(self, key: str) -> str:
        shard = random.randint(0, self.shards - 1)
        return self._store.get(self._shard_key(key, shard)) or fetch_from_redis(key)


# ---------------------------------------------------------------------------
# 4. Request Coalescing
# ---------------------------------------------------------------------------
class CoalescingCache:
    """
    Concurrent requests for the same missing key are collapsed into one fetch.
    All waiters share the result — only 1 upstream call is made per key.
    """

    def __init__(self):
        self._store: dict[str, str] = {}
        self._in_flight: dict[str, threading.Event] = {}
        self._lock = threading.Lock()
        self.fetch_count = 0   # track how many upstream fetches happened

    def read(self, key: str) -> str:
        # Fast path: already cached
        if key in self._store:
            return self._store[key]

        with self._lock:
            # Recheck inside lock
            if key in self._store:
                return self._store[key]

            if key in self._in_flight:
                # Another thread is already fetching — wait on its event
                event = self._in_flight[key]
                wait = True
            else:
                # This thread wins the fetch
                event = threading.Event()
                self._in_flight[key] = event
                wait = False

        if wait:
            event.wait()
            return self._store[key]

        # Perform the single upstream fetch
        value = fetch_from_redis(key)
        self.fetch_count += 1

        with self._lock:
            self._store[key] = value
            del self._in_flight[key]

        event.set()   # wake all waiters
        return value


# ---------------------------------------------------------------------------
# Demo
# ---------------------------------------------------------------------------
print("=== 1. Key Replication — load spread across replicas ===")
kr = KeyReplicationCache(replicas=3)
kr.write("product:iphone15", "iPhone 15 Pro")
dist = kr.shard_distribution("product:iphone15", reads=300)
for rk, count in dist.items():
    print(f"  {rk:<35} → {count} reads ({count/3:.0f}%)")

print("\n=== 2. Local L1 Cache — Redis bypassed for hot keys ===")
l1 = LocalL1Cache(l1_ttl=2.0)
l1.write("config:feature_flags", "{'dark_mode': True}")
for i in range(4):
    val, layer = l1.read("config:feature_flags")
    print(f"  Read {i+1}: '{val}'  source={layer}")

print("\n=== 3. Virtual Shards — key scattered across shards ===")
vs = VirtualShardCache(shards=4)
vs.write("product:iphone15", "iPhone 15 Pro")
shard_hits: dict[str, int] = {}
for _ in range(200):
    key = random.choice([f"product:iphone15#{i}" for i in range(4)])
    shard_hits[key] = shard_hits.get(key, 0) + 1
print("  Shard hit distribution (200 reads):")
for k, v in sorted(shard_hits.items()):
    print(f"    {k:<25} → {v}")

print("\n=== 4. Request Coalescing — N callers → 1 upstream fetch ===")
cc = CoalescingCache()
threads = [threading.Thread(target=cc.read, args=("product:iphone15",)) for _ in range(20)]
for t in threads: t.start()
for t in threads: t.join()
print(f"  20 concurrent requests → {cc.fetch_count} upstream Redis fetch(es)")
print(f"  Result: '{cc._store.get('product:iphone15')}'")
